# Final Project - ST - 554 Big data analysis

## Author: Yefrid Cordoba

### Fitting model

In this section we are going to use the whole data from the file `'power_ml_data.csv'` for the fitting of the models, using first a principal component analysis to pick the most important features, which are going to fe fed to a elastic net model crossvalidation.

We import the modules that are going to be used

In [66]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler


And create the spark session

In [67]:
spark = SparkSession.builder.getOrCreate()

Now we import the data as a pandas dataframe, and then we are going to convert it to a spark dataframe.

In [75]:
power_fit = pd.read_csv('power_ml_data.csv')

In [76]:
#We convert to spark dataframe
power_fit = spark.createDataFrame(power_fit)

In [78]:
power_fit.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [77]:
power_fit.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We are going to transformthe hour column from a `LongType()` to a `DoubleType()`.

In [79]:
type_double = SQLTransformer(
    statement="""
        SELECT * EXCEPT (Hour), 
        CAST(Hour AS DOUBLE) AS Hour 
        FROM __THIS__
    """
)

Also, we want to binarize the Hour column based on it being less than 6.5 or not.

In [101]:
 bin_hour = SQLTransformer( 
    statement = """
            SELECT *,
            CASE WHEN Hour < 6.5 THEN 1 ELSE 0 END AS bin_Hour
            FROM __THIS__
            """
 )

In [102]:
bin_hour.transform(type_double.transform(power_fit)).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|bin_Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1| 0.0|       1|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1| 0.0|       1|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1| 0.0|       1|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1| 0.0|       1|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 1

Now we ar going to 